In [ ]:
from tqdm.auto import tqdm
from data_loader import get_data
from evaluation import main_evaluation, testing
import debug

from models.gbm import GBM
from models.gmm import GMM_2
from models.msm import MSM_2
from models.heston import Heston

stocks = ['GAZP', 'SBER', 'GMKN', 'RTKM', 'MGNT']

### Описание двухкомпонентной модели гауссовой смеси (GMM_2)

#### Параметры модели
Набор оцениваемых параметров модели обозначается как $\theta = \{\pi_k, \mu_k, \sigma_k^2\}_{k=0}^1$:
* **$\pi_k$** — априорные вероятности (веса) компонентов, где $\sum_{k=0}^1 \pi_k = 1$.
* **$\mu_k$** — математические ожидания компонентов смеси.
* **$\sigma_k^2$** — дисперсии компонентов смеси (упорядочены по условию $\sigma_0^2 \le \sigma_1^2$).

---

#### Функция плотности вероятности (PDF)
Плотность вероятности одномерной двухкомпонентной модели гауссовой смеси:
$$p(x \mid \theta) = \sum_{k=0}^{1} \pi_k \mathcal{N}(x \mid \mu_k, \sigma_k^2)$$

где функция плотности нормального распределения имеет вид:
$$\mathcal{N}(x \mid \mu_k, \sigma_k^2) = \frac{1}{\sqrt{2\pi\sigma_k^2}} \exp\left(-\frac{(x - \mu_k)^2}{2\sigma_k^2}\right)$$

---

#### Формулы шагов EM-алгоритма

##### E-шаг (Expectation)
Расчет апостериорных вероятностей (весов ответственности) $\gamma_{ik}$ для каждого наблюдения $x_i$:
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)}{\sum_{j=0}^{1} \pi_j \mathcal{N}(x_i \mid \mu_j, \sigma_j^2)}$$

##### M-шаг (Maximization)
Пересчет параметров модели на основе эффективного количества наблюдений $N_k = \sum_{i=1}^{n} \gamma_{ik}$:

* **Обновление весов компонентов:**
$$\pi_k = \frac{N_k}{n}$$

* **Обновление математических ожиданий:**
$$\mu_k = \frac{\sum_{i=1}^{n} \gamma_{ik} x_i}{N_k}$$

* **Обновление дисперсий:**
$$\sigma_k^2 = \frac{\sum_{i=1}^{n} \gamma_{ik} (x_i - \mu_k)^2}{N_k}$$

---

#### Интегральная функция распределения (CDF)
Функция распределения смеси рассчитывается как кумулятивная сумма взвешенных распределений ее компонентов:
$$F(x \mid \theta) = \sum_{k=0}^{1} \pi_k \Phi\left(\frac{x - \mu_k}{\sigma_k}\right)$$

где $\Phi(z)$ — интегральная функция стандартного нормального распределения:
$$\Phi(z) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{z} \exp\left(-\frac{t^2}{2}\right) dt$$

In [ ]:
model_gmm = GMM_2()

with tqdm() as pbar:
    for st in stocks:
        tqdm.write(f"Stock: {st}")
        log_profit = get_data(st)
        
        I = main_evaluation(model_gmm, log_profit, pbar=pbar)
        
        testing(I)
        print()

0it [00:00, ?it/s]

Stock: GAZP
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.242
Kristofferson test: 3 [27.27%] | p-val = 0.109

Stock: SBER
Total periods: 11
Kupiec test: 8 [72.73%] | p-val = 0.229
Kristofferson test: 3 [27.27%] | p-val = 0.192

Stock: GMKN
Total periods: 11
Kupiec test: 10 [90.91%] | p-val = 0.339
Kristofferson test: 1 [9.09%] | p-val = 0.066

Stock: RTKM
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.363
Kristofferson test: 1 [9.09%] | p-val = 0.069

Stock: MGNT
Total periods: 11
Kupiec test: 7 [63.64%] | p-val = 0.206
Kristofferson test: 4 [36.36%] | p-val = 0.156



### Описание двухкомпонентной модели переключения режимов Маркова (MSM_2)

#### Параметры модели
Набор параметров модели обозначается как $\theta = \{A, \mu_k, \sigma_k^2\}_{k=0}^1$:
* **$A$** — матрица вероятностей переходов размера $2 \times 2$, где элемент $A_{jk} = P(S_t = k \mid S_{t-1} = j)$ задает вероятность перехода из состояния $j$ в состояние $k$.
* **$\pi$** — вектор стационарных (начальных) вероятностей состояний, рассчитываемый из матрицы переходов:
  $$\pi_0 = \frac{A_{10}}{A_{10} + A_{01}}, \quad \pi_1 = \frac{A_{01}}{A_{10} + A_{01}}$$
* **$\mu_k$** — математические ожидания, соответствующие состояниям $k \in \{0, 1\}$.
* **$\sigma_k^2$** — дисперсии, соответствующие состояниям $k \in \{0, 1\}$ (упорядочены как $\sigma_0^2 \le \sigma_1^2$).

---

#### Плотность распределения модели (Marginal Density)
Совместная маргинальная плотность полной последовательности наблюдений $X = \{x_1, x_2, \dots, x_T\}$ при заданных скрытых состояниях $S = \{S_1, S_2, \dots, S_T\}$ имеет вид:
$$p(X \mid \theta) = \sum_{S_1} \dots \sum_{S_T} p(x_1, \dots, x_T, S_1, \dots, S_T \mid \theta) = \sum_{S_1} \dots \sum_{S_T} \pi_{S_1} \mathcal{N}(x_1 \mid \mu_{S_1}, \sigma_{S_1}^2) \prod_{t=2}^{T} A_{S_{t-1} S_t} \mathcal{N}(x_t \mid \mu_{S_t}, \sigma_{S_t}^2)$$

---

#### Формулы шагов EM-алгоритма (Алгоритм Баума-Велша)

##### 1. E-шаг (Вычисление прямых и обратных переменных)

* **Прямые переменные (Forward variables) $\alpha_t(k) = p(x_1, \dots, x_t, S_t = k \mid \theta)$:**

База рекурсии: $\alpha_1(k) = \pi_k \mathcal{N}(x_1 \mid \mu_k, \sigma_k^2)$

Шаг рекурсии: $\alpha_t(k) = \mathcal{N}(x_t \mid \mu_k, \sigma_k^2) \sum_{j=0}^{1} \alpha_{t-1}(j) A_{jk}$

* **Обратные переменные (Backward variables) $\beta_t(j) = p(x_{t+1}, \dots, x_T \mid S_t = j, \theta)$:**

База рекурсии: $\beta_T(j) = 1$

Шаг рекурсии: $\beta_t(j) = \sum_{k=0}^{1} A_{jk} \mathcal{N}(x_{t+1} \mid \mu_k, \sigma_k^2) \beta_{t+1}(k)$

* **Апостериорные вероятности состояний (Сглаженные вероятности):**
  $$\gamma_t(k) = P(S_t = k \mid X, \theta) = \frac{\alpha_t(k)\beta_t(k)}{\sum_{j=0}^1 \alpha_t(j)\beta_t(j)}$$

* **Апостериорные вероятности переходов:**
  $$\xi_t(j, k) = P(S_t = j, S_{t+1} = k \mid X, \theta) = \frac{\alpha_t(j) A_{jk} \mathcal{N}(x_{t+1} \mid \mu_k, \sigma_k^2) \beta_{t+1}(k)}{\sum_{a=0}^1 \sum_{b=0}^1 \alpha_t(a) A_{ab} \mathcal{N}(x_{t+1} \mid \mu_b, \sigma_b^2) \beta_{t+1}(b)}$$

##### 2. M-шаг (Максимизация параметров)

* **Обновление матрицы переходов:**
  $$A_{jk} = \frac{\sum_{t=1}^{T-1} \xi_t(j, k)}{\sum_{t=1}^{T-1} \sum_{m=0}^1 \xi_t(j, m)}$$

* **Обновление математических ожиданий:**
  $$\mu_k = \frac{\sum_{t=1}^{T} \gamma_t(k) x_t}{\sum_{t=1}^{T} \gamma_t(k)}$$

* **Обновление дисперсий:**
  $$\sigma_k^2 = \frac{\sum_{t=1}^{T} \gamma_t(k) (x_t - \mu_k)^2}{\sum_{t=1}^{T} \gamma_t(k)}$$

---

#### Предиктивная функция распределения (Predictive CDF)
Для вычисления интегральной функции распределения на шаге $t$ используются фильтрованные вероятности состояний, полученные на основе исторической информации $\mathcal{I}_{t-1} = \{x_1, \dots, x_{t-1}\}$:

1. Вычисляется одношаговый прогноз вероятностей состояний:
   $$\hat{p}_t(k) = P(S_t = k \mid \mathcal{I}_{t-1}) = \sum_{j=0}^{1} P(S_{t-1} = j \mid \mathcal{I}_{t-1}) A_{jk}$$

2. Значение предиктивной функции распределения находится как взвешенная сумма CDF нормальных распределений:
   $$F(x_t \mid \mathcal{I}_{t-1}) = \sum_{k=0}^{1} \hat{p}_t(k) \Phi\left(\frac{x_t - \mu_k}{\sigma_k}\right)$$

где $\Phi(z)$ — интегральная функция стандартного нормального распределения.

In [18]:
model_msm = MSM_2()

with tqdm() as pbar:
    for st in stocks:
        tqdm.write(f"Stock: {st}")
        log_profit = get_data(st)
        
        I = main_evaluation(model_msm, log_profit, pbar=pbar) 
        
        testing(I)
        print()

0it [00:00, ?it/s]

Stock: GAZP
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.241
Kristofferson test: 3 [27.27%] | p-val = 0.114

Stock: SBER
Total periods: 11
Kupiec test: 8 [72.73%] | p-val = 0.229
Kristofferson test: 2 [18.18%] | p-val = 0.094

Stock: GMKN
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.326
Kristofferson test: 2 [18.18%] | p-val = 0.097

Stock: RTKM
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.307
Kristofferson test: 3 [27.27%] | p-val = 0.078

Stock: MGNT
Total periods: 11
Kupiec test: 8 [72.73%] | p-val = 0.295
Kristofferson test: 4 [36.36%] | p-val = 0.105



### Описание стохастической модели волатильности Хестона (Heston)

#### Стохастические дифференциальные уравнения модели
Непрерывное представление модели Хестона для лог-доходностей $r_t$ и скрытой дисперсии $v_t$ задается системой стохастических дифференциальных уравнений (СДУ):
$$dr_t = \left(\mu - \frac{1}{2}v_t\right)dt + \sqrt{v_t}dW_t^s$$
$$dv_t = \kappa(\theta - v_t)dt + \xi\sqrt{v_t}dW_t^v$$

Где компоненты броуновского движения скоррелированы с коэффициентом $\rho$ (зафиксирован для стабильности модели):
$$d\langle W^s, W^v \rangle_t = \rho dt, \quad \rho = -0.4$$

Параметры модели:
* $\mu$ — ожидаемая доходность (дрейф).
* $\kappa$ — скорость возврата дисперсии к среднему значению.
* $\theta$ — долгосрочная средняя дисперсия.
* $\xi$ — волатильность волатильности.

---

#### Условная плотность лог-доходностей
В дискретной реализации модели, если известена траектория скрытых волатильностей, условное распределение лог-доходности $r_t$ является нормальным:
$$p(r_t \mid v_t, \epsilon_t) = \mathcal{N}(r_t \mid m_t, s_t^2)$$

Параметры условного распределения рассчитываются с учетом коэффициента корреляции $\rho$:
$$m_t = \mu dt - \frac{1}{2}v_t dt + \rho \sqrt{v_t dt} \epsilon_t$$
$$s_t^2 = (1 - \rho^2)v_t dt$$

Функция плотности имеет вид:
$$p(r_t \mid v_t, \epsilon_t) = \frac{1}{\sqrt{2\pi s_t^2}} \exp\left(-\frac{(r_t - m_t)^2}{2s_t^2}\right)$$

---

### Алгоритм фильтра частиц

Пусть $M$ — общее число частиц, а $v_t^{(m)}$ и $W_t^{(m)}$ — значение скрытой дисперсии и нормализованный вес $m$-й частицы на шаге $t$ соответственно.

#### 1. Инициализация (t = 0)
Частицы распределяются равномерно по заданной сетке возможных значений волатильности:
$$v_0^{(m)} \in [0.001, 0.3], \quad m = 1, \dots, M$$
Всем частицам присваиваются равные начальные веса:
$$W_0^{(m)} = \frac{1}{M}$$

#### 2. Основной итерационный цикл (для каждого шага t = 1, ..., T)

##### Шаг А. Эволюция состояния (Prediction)
Для каждой частицы генерируется независимый стандартный нормальный шок $\epsilon_t^{(m)} \sim \mathcal{N}(0, 1)$. Новое состояние дисперсии рассчитывается по дискретной схеме модели Хестона:
$$v_t^{(m)} = \max\left(v_{t-1}^{(m)} + \kappa(\theta - v_{t-1}^{(m)})dt + \xi\sqrt{v_{t-1}^{(m)}dt}\epsilon_t^{(m)}, \text{floor}\right)$$

##### Шаг Б. Вычисление условной плотности (Likelihood Measurement)
Для каждой частицы вычисляется значение функции плотности вероятности текущего наблюдения $r_t$ (лог-доходности):
$$g_t^{(m)} = p(r_t \mid v_t^{(m)}, \epsilon_t^{(m)}) = \frac{1}{\sqrt{2\pi s_t^{2(m)}}} \exp\left(-\frac{(r_t - m_t^{(m)})^2}{2s_t^{2(m)}}\right)$$

Где условное математическое ожидание и условная дисперсия учитывают корреляцию параметров $\rho$:
$$m_t^{(m)} = \mu dt - \frac{1}{2}v_t^{(m)}dt + \rho\sqrt{v_t^{(m)}dt}\epsilon_t^{(m)}$$
$$s_t^{2(m)} = (1 - \rho^2)v_t^{(m)}dt$$

##### Шаг В. Обновление и нормализация весов (Weight Update)
1. Вычисляются новые ненормализованные веса частиц путем умножения априорных весов на полученные значения функции правдоподобия:
   $$W_t^{*(m)} = W_{t-1}^{(m)} \cdot g_t^{(m)}$$

2. Рассчитывается условное правдоподобие текущего шага (сумма ненормализованных весов):
   $$L_t = \sum_{m=1}^{M} W_t^{*(m)}$$

3. Производится нормализация весов, чтобы их сумма равнялась единице:
   $$W_t^{(m)} = \frac{W_t^{*(m)}}{L_t} = \frac{W_{t-1}^{(m)} \cdot g_t^{(m)}}{\sum_{j=1}^{M} W_{t-1}^{(j)} \cdot g_t^{(j)}}$$

##### Шаг Г. Накопление полного лог-правдоподобия
Общее логарифмическое правдоподобие всей выборки обновляется аддитивно:
$$\ln L(X) = \sum_{t=1}^{T} \ln L_t = \sum_{t=1}^{T} \ln \left( \sum_{m=1}^{M} W_{t-1}^{(m)} \cdot g_t^{(m)} \right)$$

##### Шаг Д. Проверка критерия пересэмплирования (Resampling)
Вычисляется эффективное число частиц (Effective Sample Size, ESS):
$$ESS = \frac{1}{\sum_{m=1}^{M} (W_t^{(m)})^2}$$

* **Если $ESS < \frac{M}{2}$:** Выполняется систематическое пересэмплирование (множественное копирование частиц с большими весами и исключение частиц с малыми весами). Новые веса всех частиц сбрасываются до однородных:
  $$W_t^{(m)} = \frac{1}{M}, \quad \forall m \in \{1, \dots, M\}$$
* **Если $ESS \ge \frac{M}{2}$:** Пересэмплирование не требуется, частицы и их нормализованные веса $W_t^{(m)}$ переходят на следующий шаг $t+1$ в неизменном виде.

---

#### Калибровка модели методами Байесовской оптимизации
Калибровка параметров выполняется с помощью функции `skopt.gp_minimize`, реализующей оптимизацию на основе Гауссовых процессов с функцией полезности Expected Improvement (EI).

Целевая функция минимизирует отрицательное лог-правдоподобие с добавлением двух типов ограничений:
1. **Штраф за нарушение условия Феллера:** Если $2\kappa\theta - \xi^2 \le 0$, накладывается строгий штраф, гарантирующий стремление процесса дисперсии к строго положительным значениям:
   $$\text{Penalty}_{\text{Feller}} = 10^6 + \left|2\kappa\theta - \xi^2\right| \cdot 10^5$$
2. **L2-регуляризация параметров:** Для предотвращения нестабильности оценок применяется штрафное смещение к априорным центрам параметров $\theta_0 = [0.0, 3.0, 0.05, 0.4]$ с весами $\lambda = [0.1, 0.2, 10.0, 0.5]$:
   $$\text{Penalty}_{\text{Reg}} = \sum_{j} \lambda_j (\theta_j - \theta_{0,j})^2$$

Итоговый оптимизируемый функционал: $Q(\theta) = -\ln L(Y) + \text{Penalty}_{\text{Feller}} + \text{Penalty}_{\text{Reg}}$.

---

#### Расчет предиктивной функции распределения (Predictive CDF)
Предиктивная функция распределения $F(r_t \mid \mathcal{I}_{t-1})$ строится на основе фильтра частиц, прошедшего фазу прогрева (последовательного обновления внутренних состояний весов и частиц на обучающей\ предшествующей выборке).

Для каждого тестового наблюдения $r_t$ расчет производится следующим образом:
$$F(r_t \mid \mathcal{I}_{t-1}) = \sum_{m=1}^{M} W_{t-1}^{(m)} \Phi\left(\frac{r_t - \text{pred\_m}_t^{(m)}}{\sqrt{\text{pred\_s2}_t^{(m)}}}\right)$$

Где параметры безусловного (по отношению к новому шоку) распределения равны:
$$\text{pred\_m}_t^{(m)} = \mu dt - \frac{1}{2}v_{t-1}^{(m)}dt$$
$$\text{pred\_s2}_t^{(m)} = v_{t-1}^{(m)}dt$$

In [19]:
model_heston = Heston(n_particles=10000)

with tqdm() as pbar:
    for st in stocks:
        tqdm.write(f"Stock: {st}")
        log_profit = get_data(st)
        
        I = main_evaluation(model_heston, log_profit, pbar=pbar) 
        
        testing(I)
        print()

0it [00:00, ?it/s]

Stock: GAZP
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.29
Kristofferson test: 2 [18.18%] | p-val = 0.094

Stock: SBER
Total periods: 11
Kupiec test: 9 [81.82%] | p-val = 0.278
Kristofferson test: 3 [27.27%] | p-val = 0.108

Stock: GMKN
Total periods: 11
Kupiec test: 8 [72.73%] | p-val = 0.295
Kristofferson test: 4 [36.36%] | p-val = 0.155

Stock: RTKM
Total periods: 11
Kupiec test: 10 [90.91%] | p-val = 0.359
Kristofferson test: 4 [36.36%] | p-val = 0.075

Stock: MGNT
Total periods: 11
Kupiec test: 7 [63.64%] | p-val = 0.283
Kristofferson test: 4 [36.36%] | p-val = 0.169

